# Dataset Viewer

Drag the slider to flip through any dataset, one image at a time.

Works with MNIST, FashionMNIST, CIFAR-10, KITTI, COCO, INRIA Person — or any torchvision-style dataset where indexing returns an `(image, label)` pair.

## Step 1 — Load a dataset

Remove the `#` in front of exactly one `dataset = ...` line and run the cell. The comment above each line says what it is and what you can change. The rest of the notebook only needs a variable called `dataset`.


In [ ]:
from torchvision import datasets, transforms

# transform = what happens to each image before you see it. Swap in one of these to change it:
#   transforms.ToTensor()
#       -> the original image, unchanged (the default)
#   transforms.Compose([transforms.Grayscale(), transforms.ToTensor()])
#       -> colour datasets shown in grey
#   transforms.Compose([transforms.Resize((64, 64)), transforms.ToTensor()])
#       -> every image squashed to 64x64 (on KITTI and COCO the boxes will then no longer line up)
transform = transforms.ToTensor()

# --- Pick ONE: remove the # in front of exactly one `dataset = ...` line (and put a # back on the old one) ---

# MNIST: 28x28 handwritten digits, 10 classes.
#   train=True -> 60,000 images; train=False -> the 10,000 test images instead.
dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

# FashionMNIST: 28x28 photos of clothes (T-shirt/top, trouser, sneaker, bag...), 10 classes. Same counts as MNIST.
# dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)

# CIFAR-10: 32x32 colour photos (airplane, automobile, bird, cat...), 10 classes.
#   train=True -> 50,000 images; train=False -> 10,000 test images.
# dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)

# KITTI: road scenes from a car's camera, with a box around every car, pedestrian, cyclist...
#   train=True -> 7,481 images with boxes; train=False -> 7,518 test images with NO boxes.
#   Downloads about 12.6 GB the first time.
# dataset = datasets.Kitti(root='./data', train=True, download=True, transform=transform)

# COCO: everyday photos with a box around each object (80 kinds: person, car, dog, cup...).
#   Run the COCO download cell in Tips first. Change 'val2017' to 'train2017' in BOTH places
#   -> 118,287 images instead of 5,000 (download train2017.zip instead; much bigger).
#   Change 'instances_' to 'person_keypoints_' -> only people get boxes.
# dataset = datasets.CocoDetection('./data/val2017', './data/annotations/instances_val2017.json', transform=transform)

# INRIA Person: street photos in two classes, 'pos' (people in it) and 'neg' (no people).
#   Run the INRIA download cell in Tips first.
#   Change 'Train' to 'Test' -> 288 people photos + 453 empty ones, instead of 614 + 1,218.
#   It also lists an empty third class, 'annotations' (a folder of box files); ignore it.
# dataset = datasets.ImageFolder('./data/INRIAPerson/Train', transform=transform, allow_empty=True)

print(f'Loaded {len(dataset)} images')
print(f'Classes: {getattr(dataset, "classes", "boxes per image (detection)")}')

## Step 2 — Browse the images

Run the cell below. A control panel will appear with:

- **⏮ First** and **Last ⏭** — jump to the very first / very last image
- **← Previous** and **Next →** — step one image at a time
- **Random** — pick a random image (great for skimming the dataset)
- **Go to:** — type any image number into the box, then **click the blue Go button** to jump there

A counter shows how many images are in the dataset and which one you're looking at.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

total = len(dataset)
current_idx = 0

# --- Build the controls ---
first_btn  = widgets.Button(description='⏮ First',     layout=widgets.Layout(width='100px'))
prev_btn   = widgets.Button(description='← Previous',  layout=widgets.Layout(width='120px'))
next_btn   = widgets.Button(description='Next →',      layout=widgets.Layout(width='120px'))
last_btn   = widgets.Button(description='Last ⏭',      layout=widgets.Layout(width='100px'))
random_btn = widgets.Button(description='Random',      layout=widgets.Layout(width='100px'))

idx_input  = widgets.IntText(value=0, description='Go to:',
                             layout=widgets.Layout(width='180px'))
go_btn     = widgets.Button(description='Go', button_style='primary',
                            layout=widgets.Layout(width='60px'))

counter    = widgets.HTML(value='')
picture    = widgets.Output()

def render(idx):
    global current_idx
    # Clamp to a valid range so a bad input can't crash anything
    idx = max(0, min(int(idx), total - 1))
    current_idx = idx

    counter.value = (
        f"<b>Image {idx + 1:,} of {total:,}</b> "
        f"<span style='color:#888'>(index {idx})</span>"
    )

    image, label = dataset[idx]
    if isinstance(image, torch.Tensor):
        image = image.numpy()
        if image.ndim == 3:
            image = image.transpose(1, 2, 0)
    image = np.squeeze(image)
    cmap = 'gray' if image.ndim == 2 else None

    with picture:
        clear_output(wait=True)
        plt.close('all')                    # avoid stacking old figures
        h, w = image.shape[:2]
        # Picture size in inches: change the 4s to make it bigger or smaller; 12 caps the width of wide images.
        fig, ax = plt.subplots(figsize=(min(4 * w / h, 12), 4))
        ax.imshow(image, cmap=cmap)
        if isinstance(label, list):         # detection (KITTI, COCO): a list of boxes
            for obj in label:
                if 'type' in obj:           # KITTI box: [x1, y1, x2, y2]
                    x1, y1, x2, y2 = obj['bbox']
                    name = obj['type']
                else:                       # COCO box: [x, y, width, height]
                    x1, y1, bw, bh = obj['bbox']
                    x2, y2 = x1 + bw, y1 + bh
                    name = dataset.coco.cats[obj['category_id']]['name']
                # Box style: color='lime' -> try 'red' or 'yellow'; add linewidth=2 for thicker boxes.
                ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color='lime'))
                # Box name: delete this line to hide the names; raise fontsize=8 for bigger text.
                ax.text(x1, y1, name, color='lime', fontsize=8, va='bottom')
            ax.set_title(f'{len(label)} objects')
        elif label is None:                 # KITTI test split has no labels
            ax.set_title('No labels')
        else:
            class_name = dataset.classes[label] if hasattr(dataset, 'classes') else label
            ax.set_title(f'Label: {label}  ({class_name})')
        ax.axis('off')
        plt.show()

# --- Wire up the buttons ---
def on_first(b):  render(0)
def on_prev(b):   render((current_idx - 1) % total)
def on_next(b):   render((current_idx + 1) % total)
def on_last(b):   render(total - 1)
def on_random(b): render(int(np.random.randint(0, total)))
def on_go(b):     render(idx_input.value)

first_btn.on_click(on_first)
prev_btn.on_click(on_prev)
next_btn.on_click(on_next)
last_btn.on_click(on_last)
random_btn.on_click(on_random)
go_btn.on_click(on_go)

# --- Lay it out and show ---
row1 = widgets.HBox([first_btn, prev_btn, next_btn, last_btn, random_btn])
row2 = widgets.HBox([idx_input, go_btn, counter])
display(widgets.VBox([row1, row2, picture]))
render(0)  # first image shown: change 0 to start somewhere else, e.g. render(100)

### Tips

- **Switch dataset:** go back to Step 1, put a `#` back on the current line, remove it from another, run it, then re-run Step 2.
- **COCO download** (about 1.07 GB: the 5,000 val2017 images plus annotations). Run once in a cell:
  ```
  !pip install pycocotools
  !mkdir -p data && cd data && wget -q http://images.cocodataset.org/zips/val2017.zip && unzip -q val2017.zip
  !cd data && wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip && unzip -q annotations_trainval2017.zip
  ```
- **INRIA Person download** (about 1 GB, a copy hosted by OpenIMAJ; the original INRIA site is often down). Run once in a cell:
  ```
  !mkdir -p data && cd data && wget -q http://datasets.openimaj.org/INRIAPerson.zip && unzip -q INRIAPerson.zip
  ```
- **Your own dataset:** as long as `dataset[i]` returns `(image, label)` and `len(dataset)` works, the viewer will handle it — grayscale or colour, tensors or PIL images.
